# scWAT Xenium slide-level QC summary

Aggregate the four independently processed sections without treating cells as biological replicates.

## Goal

Verify complete Region 1-4 coverage and answer four technical-QC questions: direct alarm/candidate-gene evidence, subset-versus-full review burden, spatial clustering/morphology-review targets, and within-mouse section concordance.

## Setup

### Parameters

In [ ]:
PROJECT_ROOT <- "/dssg/home/acct-svetoslav_chakarov/svetoslav_chakarov/Lab_members/Yanan_Hu"
PIPELINE_REPO <- file.path(PROJECT_ROOT, "adipose_analysis", "YNH_Xenium_scWAT")
RUN_LABEL <- "full_notebook_qc_v1"
EXPECTED_SECTION_COUNT <- 4L
METADATA_PATH <- file.path(PIPELINE_REPO, "config", "scwat_sample_manifest.tsv")
EXTENDED_QC_CONFIG_PATH <- file.path(PIPELINE_REPO, "config", "extended_qc_defaults.tsv")
SUBSET_REFERENCE_PATH <- file.path(PIPELINE_REPO, "config", "subset_qc_reference.tsv")


In [ ]:
RUN_ROOT <- file.path(PROJECT_ROOT, "adipose_analysis", "scwat_qc_outputs", RUN_LABEL)
source(file.path(PIPELINE_REPO, "R", "source.R"))
for (package in c("Matrix", "jsonlite", "ggplot2")) require_package(package)
assert_path_within(PROJECT_ROOT, RUN_ROOT)
assert_path_within(PROJECT_ROOT, tempdir())
stopifnot(EXPECTED_SECTION_COUNT == 4L)
cat("Slide QC run root:", RUN_ROOT, "\n")


## Inputs

Exactly four independently completed core and extended section bundles are required. The verified manifest defines Mouse 1 (62308/62309) and Mouse 2 (62310/62311); mouse is the biological replicate and section is a technical processing unit.

## Completeness Checks

In [ ]:
coverage <- validate_four_section_outputs(RUN_ROOT, paste0("Region_", seq_len(EXPECTED_SECTION_COUNT)))
extended_coverage <- validate_four_extended_section_outputs(RUN_ROOT, coverage$region_id)
stopifnot(identical(coverage$region_id, extended_coverage$region_id))
list(core = coverage, extended = extended_coverage)


## QC Results

In [ ]:
slide_data <- read_slide_qc_outputs(RUN_ROOT, coverage$region_id)
slide_summary <- summarise_slide_qc(slide_data)
extended_slide_data <- read_extended_slide_qc_outputs(RUN_ROOT, coverage$region_id)
manifest <- utils::read.delim(METADATA_PATH, check.names = FALSE)
extended_config <- read_extended_qc_config(EXTENDED_QC_CONFIG_PATH)
subset_reference <- utils::read.delim(SUBSET_REFERENCE_PATH, check.names = FALSE)
extended_slide_summary <- summarise_extended_slide_qc(extended_slide_data, slide_summary$section_summary, manifest, extended_config, subset_reference)
slide_summary$section_summary


## Question 1 - Which alarms and candidate genes are affected?

The alarm table reports directly available 10x evidence. Candidate genes are ranked from cross-section abundance and transcript-QV patterns, remain `CANDIDATE_NOT_CONFIRMED`, and cannot identify the exact cycle; cycle identity requires 10x diagnostics.

In [ ]:
extended_slide_data$cycle_alarm_evidence
candidate_display <- extended_slide_summary$candidates[order(extended_slide_summary$candidates$evidence_tier, extended_slide_summary$candidates$gene), , drop = FALSE]
candidate_display[seq_len(min(30L, nrow(candidate_display))), , drop = FALSE]


## Question 2 - Does full-data QC reproduce the subset ranking?

The comparison is descriptive across four technical sections. `NOT_RUN_LOCAL_SUBSET` means this question remains pending until the full-HPC run.

In [ ]:
extended_slide_summary$ranking
extended_slide_summary$rank_agreement


## Question 3 - Are review flags spatially clustered?

Global kNN clustering, tissue-edge proxies, dense-aggregate proxies, and candidate hotspot bins are coordinate-based diagnostics. Hotspots are not labelled folds or tears without morphology/image review.

In [ ]:
extended_slide_data$spatial_global
extended_slide_data$spatial_edge_density
extended_slide_data$manual_review_manifest


## Question 4 - Are the two sections from each mouse technically concordant?

The two sections per mouse are compared as technical pairs. Thresholds are advisory, and these results are not biological hypothesis tests.

In [ ]:
extended_slide_summary$concordance$summary


## Cell-style Figures

Colors are fixed across sections; distributions are descriptive and do not imply cell-level biological replication.

In [ ]:
slide_plots <- plot_slide_qc(slide_data, slide_summary)
extended_slide_plots <- plot_extended_slide_qc(extended_slide_data, extended_slide_summary)
for (plot in slide_plots) print(plot)
for (plot in extended_slide_plots) print(plot)


## Readiness

The original Phase 0-2 gates remain authoritative and unchanged. The worst section gate determines slide readiness; unresolved Xenium errors still block biology.

In [ ]:
slide_summary$readiness
extended_slide_summary$status
cat("Overall slide QC status:", slide_summary$overall_status, "\n")


## Outputs

In [ ]:
slide_artifacts <- write_slide_qc_artifacts(PROJECT_ROOT, RUN_ROOT, slide_data, slide_summary, slide_plots)
extended_slide_artifacts <- write_extended_slide_qc_artifacts(PROJECT_ROOT, RUN_ROOT, extended_slide_data, extended_slide_summary, extended_slide_plots)
saved_summary <- readRDS(file.path(RUN_ROOT, "slide_summary", "slide_qc_summary.rds"))
stopifnot(nrow(saved_summary$data$coverage) == 4L)
stopifnot(length(unique(saved_summary$data$cell_metadata$region_id)) == 4L)
stopifnot(validate_extended_slide_qc_artifacts(RUN_ROOT, stop_on_error = TRUE))
list(core = data.frame(artifact = basename(slide_artifacts), path = slide_artifacts),
     extended = data.frame(artifact = basename(extended_slide_artifacts), path = extended_slide_artifacts))
